In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from ipywidgets import FloatSlider, Checkbox, Button, VBox, HBox, Output, HTML
from IPython.display import display, clear_output
import warnings
warnings.filterwarnings('ignore')

# Налаштування matplotlib
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

print("✓ Всі бібліотеки імпортовані успішно!")

✓ Всі бібліотеки імпортовані успішно!


In [2]:
class HarmonicGenerator:
    """Генератор гармонічних сигналів з шумом та фільтрацією"""
    
    def __init__(self):
        self.t = np.linspace(0, 10, 1000)
        self.reset_params()
    
    def reset_params(self):
        """Скидання параметрів до початкових значень"""
        self.amplitude = 1.0
        self.frequency = 1.0
        self.phase = 0.0
        self.noise_mean = 0.0
        self.noise_std = 0.1
        self.generate_noise()
    
    def generate_noise(self):
        """Генерація нового шуму"""
        self.noise = np.random.normal(self.noise_mean, self.noise_std, len(self.t))
    
    def harmonic(self, amplitude, frequency, phase):
        """Генерація чистої гармоніки: y(t) = A*sin(ωt + φ)"""
        return amplitude * np.sin(2 * np.pi * frequency * self.t + phase)
    
    def apply_filter(self, signal_data, cutoff_freq):
        """Застосування фільтра Баттерворта (lowpass)"""
        nyquist = 0.5 * (len(self.t) / (self.t[-1] - self.t[0]))
        normal_cutoff = min(cutoff_freq / nyquist, 0.99)
        b, a = signal.butter(4, normal_cutoff, btype='low', analog=False)
        return signal.filtfilt(b, a, signal_data)

print("✓ HarmonicGenerator готовий!")

✓ HarmonicGenerator готовий!


In [3]:
class InteractiveHarmonic:
    """Інтерактивна візуалізація гармоніки з шумом та фільтрацією"""
    
    def __init__(self):
        self.gen = HarmonicGenerator()
        self.last_noise_params = (0.0, 0.1)
        self.output = Output()
        self.create_widgets()

    def create_widgets(self):
        """Створення інтерактивних елементів"""
        style = {'description_width': '120px'}
        layout_s = {'width': '350px'}

        # Слайдери для параметрів гармоніки
        self.amp_slider = FloatSlider(
            value=1.0, min=0.1, max=2.0, step=0.1,
            description='Амплітуда:', style=style, layout=layout_s
        )
        self.freq_slider = FloatSlider(
            value=1.0, min=0.1, max=5.0, step=0.1,
            description='Частота:', style=style, layout=layout_s
        )
        self.phase_slider = FloatSlider(
            value=0.0, min=0.0, max=6.28, step=0.1,
            description='Фаза:', style=style, layout=layout_s
        )
        
        # Слайдери для параметрів шуму
        self.noise_mean_slider = FloatSlider(
            value=0.0, min=-0.5, max=0.5, step=0.05,
            description='Середнє шуму:', style=style, layout=layout_s
        )
        self.noise_std_slider = FloatSlider(
            value=0.1, min=0.0, max=0.5, step=0.01,
            description='Дисперсія шуму:', style=style, layout=layout_s
        )
        
        # Слайдер для фільтра
        self.cutoff_slider = FloatSlider(
            value=5.0, min=1.0, max=20.0, step=0.5,
            description='Частота зрізу:', style=style, layout=layout_s
        )

        # Чекбокси
        self.show_noise_check = Checkbox(value=True, description='Показати шум')
        self.show_filtered_check = Checkbox(value=True, description='Показати фільтр')
        
        # Кнопка Reset
        self.reset_button = Button(description=' Reset', button_style='warning', 
                                   layout={'width': '120px'})
        self.reset_button.on_click(self.reset)

        # Прив'язка обробників подій
        for slider in [self.amp_slider, self.freq_slider, self.phase_slider]:
            slider.observe(self.on_harmonic_change, names='value')
        
        for slider in [self.noise_mean_slider, self.noise_std_slider]:
            slider.observe(self.on_noise_change, names='value')
        
        for w in [self.cutoff_slider, self.show_noise_check, self.show_filtered_check]:
            w.observe(self.update_plot, names='value')

    def on_harmonic_change(self, change):
        """Обробка зміни параметрів гармоніки (шум НЕ змінюється)"""
        self.gen.amplitude = self.amp_slider.value
        self.gen.frequency = self.freq_slider.value
        self.gen.phase = self.phase_slider.value
        self.update_plot(change)

    def on_noise_change(self, change):
        """Обробка зміни параметрів шуму (генерація НОВОГО шуму)"""
        current = (round(self.noise_mean_slider.value, 5), 
                  round(self.noise_std_slider.value, 5))
        
        if current != self.last_noise_params:
            self.gen.noise_mean = current[0]
            self.gen.noise_std = current[1]
            self.gen.generate_noise()
            self.last_noise_params = current
        
        self.update_plot(change)

    def reset(self, b):
        """Скидання всіх параметрів до початкових"""
        # Тимчасово відключаємо observers
        for slider in [self.amp_slider, self.freq_slider, self.phase_slider]:
            slider.unobserve(self.on_harmonic_change, names='value')
        for slider in [self.noise_mean_slider, self.noise_std_slider]:
            slider.unobserve(self.on_noise_change, names='value')

        # Скидання генератора
        self.gen.reset_params()
        
        # Скидання віджетів
        self.amp_slider.value = 1.0
        self.freq_slider.value = 1.0
        self.phase_slider.value = 0.0
        self.noise_mean_slider.value = 0.0
        self.noise_std_slider.value = 0.1
        self.cutoff_slider.value = 5.0
        self.show_noise_check.value = True
        self.show_filtered_check.value = True
        self.last_noise_params = (0.0, 0.1)

        # Повторно підключаємо observers
        for slider in [self.amp_slider, self.freq_slider, self.phase_slider]:
            slider.observe(self.on_harmonic_change, names='value')
        for slider in [self.noise_mean_slider, self.noise_std_slider]:
            slider.observe(self.on_noise_change, names='value')

        self.update_plot(None)

    def update_plot(self, change):
        """Оновлення графіків"""
        with self.output:
            clear_output(wait=True)
            
            # Генерація сигналів
            clean = self.gen.harmonic(self.gen.amplitude, self.gen.frequency, self.gen.phase)
            noisy = clean + self.gen.noise

            # Створення фігури з двома підграфіками
            fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))

            # --- Верхній графік: Оригінальний сигнал ---
            ax1.plot(self.gen.t, clean, 'b-', linewidth=2, label='Чиста гармоніка')
            
            if self.show_noise_check.value:
                ax1.plot(self.gen.t, noisy, color='orange', linewidth=1, 
                        alpha=0.7, label='Зашумлений сигнал')
            
            ax1.set_title('Оригінальний сигнал', fontsize=14, fontweight='bold')
            ax1.set_xlabel('Час (с)', fontsize=11)
            ax1.set_ylabel('Амплітуда', fontsize=11)
            ax1.legend(loc='upper right', fontsize=10)
            ax1.grid(True, alpha=0.3)

            # --- Нижній графік: Відфільтрований сигнал ---
            if self.show_filtered_check.value:
                source = noisy if self.show_noise_check.value else clean
                filtered = self.gen.apply_filter(source, self.cutoff_slider.value)

                ax2.plot(self.gen.t, filtered, 'g-', linewidth=2, label='Відфільтрований')
                ax2.plot(self.gen.t, clean, 'b--', linewidth=1, alpha=0.5, 
                        label='Чиста (еталон)')
                
                ax2.set_title('Відфільтрований сигнал', fontsize=14, fontweight='bold')
                ax2.set_xlabel('Час (с)', fontsize=11)
                ax2.set_ylabel('Амплітуда', fontsize=11)
                ax2.legend(loc='upper right', fontsize=10)
                ax2.grid(True, alpha=0.3)
            else:
                ax2.text(0.5, 0.5, 'Відфільтрований сигнал вимкнено', 
                        ha='center', va='center', fontsize=12, 
                        transform=ax2.transAxes)
                ax2.set_xlabel('Час (с)', fontsize=11)
                ax2.set_ylabel('Амплітуда', fontsize=11)

            plt.tight_layout()
            plt.show()

    def display(self):
        """Відображення інтерфейсу"""
        title = HTML("""
        <div style='background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); 
                    padding: 15px; border-radius: 10px; margin-bottom: 15px;'>
            <h2 style='color: white; margin: 0; text-align: center;'>
                 Інтерактивна візуалізація гармоніки з шумом
            </h2>
        </div>
        """)

        instructions = HTML("""
        <div style='background: #f0f4ff; padding: 12px; border-radius: 8px; 
                    margin-bottom: 15px; font-size: 13px; border-left: 4px solid #667eea;'>
        <b> ІНСТРУКЦІЯ КОРИСТУВАЧА:</b><br><br>
        
        <b> Параметри гармоніки:</b><br>
        • <b>Амплітуда</b> — висота коливань (максимальне відхилення від нуля)<br>
        • <b>Частота</b> — кількість повних коливань за секунду<br>
        • <b>Фаза</b> — зміщення початку хвилі (0 до 2π)<br><br>
        
        <b> Параметри шуму:</b><br>
        • <b>Середнє шуму</b> — зсув шуму відносно нуля<br>
        • <b>Дисперсія шуму</b> — інтенсивність (розкид) шуму<br><br>
        
        <b> Фільтр:</b><br>
        • <b>Частота зрізу</b> — частота фільтра Баттерворта<br>
        &nbsp;&nbsp;(менше значення = більше згладжування)<br><br>
        
        <b> Чекбокси:</b><br>
        • <b>Показати шум</b> — увімкнути/вимкнути відображення шуму<br>
        • <b>Показати фільтр</b> — увімкнути/вимкнути відфільтрований сигнал<br><br>
        
        <b> Кнопка Reset</b> — повернути всі параметри до початкових значень<br><br>
        
        <b>ВАЖЛИВО:</b><br>
        • При зміні параметрів <b>ГАРМОНІКИ</b> → шум залишається незмінним<br>
        • При зміні параметрів <b>ШУМУ</b> → генерується НОВИЙ шум
        </div>
        """)

        # Організація віджетів у колонки
        col1 = VBox([
            HTML("<div style='background:#667eea; color:white; padding:8px; "
                 "border-radius:5px; text-align:center; margin-bottom:10px;'>"
                 "<b> Параметри гармоніки</b></div>"),
            self.amp_slider,
            self.freq_slider,
            self.phase_slider
        ])
        
        col2 = VBox([
            HTML("<div style='background:#f59e0b; color:white; padding:8px; "
                 "border-radius:5px; text-align:center; margin-bottom:10px;'>"
                 "<b>🔊 Параметри шуму</b></div>"),
            self.noise_mean_slider,
            self.noise_std_slider
        ])
        
        col3 = VBox([
            HTML("<div style='background:#10b981; color:white; padding:8px; "
                 "border-radius:5px; text-align:center; margin-bottom:10px;'>"
                 "<b>🔧 Фільтр та відображення</b></div>"),
            self.cutoff_slider,
            self.show_noise_check,
            self.show_filtered_check,
            self.reset_button
        ])

        controls = HBox([col1, col2, col3], 
                       layout={'justify_content': 'space-around', 
                              'margin': '10px 0'})
        
        display(VBox([title, instructions, controls, self.output]))
        self.update_plot(None)

print("✓ InteractiveHarmonic готовий!")

✓ InteractiveHarmonic готовий!


In [4]:
# Створення та запуск інтерактивного додатку
app = InteractiveHarmonic()
app.display()